In [129]:
import pandas as pd

In [130]:
import numpy as np

In [131]:
df = pd.read_csv('Titanic-Dataset.csv')

In [132]:
df.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [133]:
df['Age'] = df['Age'].fillna(df['Age'].mean())

In [134]:
df.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age              0
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [135]:
df.drop(['Ticket','Cabin','PassengerId','Name',],axis=1,inplace=True)

In [136]:
df.shape

(891, 8)

In [137]:
df.isnull().sum()

Survived    0
Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    2
dtype: int64

In [138]:
df['Embarked'] = df['Embarked'].fillna(
    df['Embarked'].mode()[0]
)

In [139]:
df.isnull().sum()

Survived    0
Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    0
dtype: int64

In [140]:
x = df.drop("Survived",axis=1)
y = df["Survived"]

In [141]:
y

0      0
1      1
2      1
3      1
4      0
      ..
886    0
887    1
888    0
889    1
890    0
Name: Survived, Length: 891, dtype: int64

In [142]:
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


In [143]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

transformer = ColumnTransformer(
    transformers=[
        (
            'model1',
            OneHotEncoder(sparse_output=False, drop='first'),
            ['Sex', 'Embarked']
        )
    ],
    remainder='passthrough'
)

In [144]:
x=transformer.fit_transform(x)

In [145]:
from sklearn.model_selection import train_test_split

In [146]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

In [147]:
!pip install keras-tuner


[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [148]:
import keras_tuner as kt 
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam

In [149]:
def build_model(hp):
    model = Sequential()
    n_hidden = hp.Int('n_hidden_layers',min_value=1,max_value=5,step=1)
    for i in range(n_hidden):
        model.add(Dense(units=hp.Int(f'units_{i}',min_value=32,max_value=512,step=32),activation='relu'))
    model.add(Dense(1, activation='sigmoid'))
    learning_rate = hp.Choice('learning_value',values=[1e-2, 1e-3, 1e-4])
    model.compile(optimizer=Adam(learning_rate=learning_rate),loss='binary_crossentropy',metrics=['accuracy'])
    return model

In [150]:
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=20,
    directory='keras_tuning',
    project_name='customer_churn'
)

Reloading Tuner from keras_tuning\customer_churn\tuner0.json


In [152]:
tuner.search(x_train,y_train,epochs=20,validation_split=0.2)


Search: Running Trial #3

Value             |Best Value So Far |Hyperparameter
4                 |1                 |n_hidden_layers
448               |512               |units_0
0.0001            |0.001             |learning_value



Traceback (most recent call last):
  File "C:\Users\ADMIN\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras_tuner\src\engine\base_tuner.py", line 274, in _try_run_and_update_trial
    self._run_and_update_trial(trial, *fit_args, **fit_kwargs)
  File "C:\Users\ADMIN\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras_tuner\src\engine\base_tuner.py", line 239, in _run_and_update_trial
    results = self.run_trial(trial, *fit_args, **fit_kwargs)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\ADMIN\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras_tuner\src\engine\tuner.py", line 309, in run_trial
    self._configure_tensorboard_dir(callbacks, trial, execution)
  File "C:\Users\ADMIN\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras_tuner\src\engine\tuner.py", line 421, in _configure_tensorboard_dir
    from tensorboard.plugins.hparams import api as hparams_api
ModuleNotFoundError: No module named 'tensorb

RuntimeError: Number of consecutive failures exceeded the limit of 3.
Traceback (most recent call last):
  File "C:\Users\ADMIN\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras_tuner\src\engine\base_tuner.py", line 274, in _try_run_and_update_trial
    self._run_and_update_trial(trial, *fit_args, **fit_kwargs)
  File "C:\Users\ADMIN\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras_tuner\src\engine\base_tuner.py", line 239, in _run_and_update_trial
    results = self.run_trial(trial, *fit_args, **fit_kwargs)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\ADMIN\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras_tuner\src\engine\tuner.py", line 309, in run_trial
    self._configure_tensorboard_dir(callbacks, trial, execution)
  File "C:\Users\ADMIN\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras_tuner\src\engine\tuner.py", line 421, in _configure_tensorboard_dir
    from tensorboard.plugins.hparams i

In [126]:
best_hp = tuner.get_best_hyperparameters()[0]

print(best_hp.values)

{'n_hidden_layers': 1, 'units_0': 512, 'learning_value': 0.001}


In [127]:
best_model = tuner.hypermodel.build(best_hp)

history = best_model.fit(
    x_train,
    y_train,
    epochs=50,
    validation_split=0.2
)

Epoch 1/50
18/18 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.6450 - loss: 0.6397 - val_accuracy: 0.6993 - val_loss: 0.5539
Epoch 2/50
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.6714 - loss: 0.6205 - val_accuracy: 0.7273 - val_loss: 0.5304
Epoch 3/50
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7047 - loss: 0.5655 - val_accuracy: 0.7203 - val_loss: 0.5912
Epoch 4/50
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7206 - loss: 0.5408 - val_accuracy: 0.7622 - val_loss: 0.5672
Epoch 5/50
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7153 - loss: 0.5418 - val_accuracy: 0.7972 - val_loss: 0.4898
Epoch 6/50
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.7873 - loss: 0.5119 - val_accuracy: 0.7413 - val_loss: 0.4954
Epoch 7/50
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7610 - loss: 0.5136 - val_accuracy: 0.7902 - val_loss: 0.4629
Epoch 8/50
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7610 - loss: 0.5039 - val_accuracy: 0.7692 - v